In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [6]:
DB_USER     = 'teamE_postgresSQL'
DB_PASSWORD = 'postgres@123'
DB_HOST     = 'team-e-postgres.postgres.database.azure.com'
DB_PORT     = '5432'
DB_NAME     = 'postgres'
db_url = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_url)

In [20]:
import psycopg2

# 建立连接
conn = psycopg2.connect(
    host="team-e-postgres.postgres.database.azure.com",
    port=5432,
    dbname="teamedb",
    user="teamE_postgresSQL",
    password="postgres@123",
    sslmode="require"
)

# 建立游标
cur = conn.cursor()

# 测试：查看当前数据库版本
cur.execute("SELECT version();")
print(cur.fetchone())

('PostgreSQL 16.9 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 11.2.0, 64-bit',)


In [14]:
import pandas as pd

# 1. 读取原始数据
df = pd.read_csv("./data/merged_data.csv")

# 2. 定义啤酒前缀和对应的 ID（对应你数据库中 product 表里的 id 顺序）
beer_prefix = ['white_beer', 'lager', 'pale_ale', 'fruit_beer', 'dark_beer', 'ipa']
id_map = {
    'white_beer': 1,
    'lager':      2,
    'pale_ale':   3,
    'fruit_beer': 4,
    'dark_beer':  5,
    'ipa':        6,
}

# 3. 分别对“瓶数”和“销售额”做 melt
vol = df.melt(
    id_vars=['date'],
    value_vars=[f"{b}_bottles" for b in beer_prefix],
    var_name='beer', value_name='volume'
)
sales = df.melt(
    id_vars=['date'],
    value_vars=[f"{b}_sales" for b in beer_prefix],
    var_name='beer', value_name='sales'
)

# 4. 去掉列名后缀，统一成啤酒前缀
vol['beer']   = vol['beer'].str.replace('_bottles', '')
sales['beer'] = sales['beer'].str.replace('_sales',   '')

# 5. 按 (date, beer) 合并 volume 和 sales
long = pd.merge(vol, sales, on=['date', 'beer'])

# 6. 根据映射把 beer 列替换成 beer_id
long['beer_id'] = long['beer'].map(id_map)

# 7. 取所需列，并按习惯重命名
result = long[['date', 'beer_id', 'volume', 'sales']].rename(columns={
    "beer_id": "product_id",
    'volume': 'sales_count',
    'sales':  'sales_revenue'
})

# 看看效果
result


,date,product_id,sales_count,sales_revenue
0,2024-04-01,1,4,3600
1,2024-04-02,1,3,2700
2,2024-04-03,1,2,1800
3,2024-04-04,1,1,900
4,2024-04-05,1,2,1800
...,...,...,...,...
1987,2025-03-27,6,2,1800
1988,2025-03-28,6,3,2700
1989,2025-03-29,6,1,900
1990,2025-03-31,6,2,1800


In [ ]:
df_long = result.copy()

In [34]:
from datetime import datetime
now = datetime.now()
df_long['creator_id']         = 1
df_long['update_id']          = 1
df_long['weather_history_id'] = 9
df_long['is_deleted']         = False
df_long['created_at']         = now
df_long['updated_at']         = now
df_long

,date,product_id,sales_count,sales_revenue,creator_id,update_id,weather_history_id,is_deleted,created_at,updated_at
0,2024-04-01,1,4,3600,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
1,2024-04-02,1,3,2700,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
2,2024-04-03,1,2,1800,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
3,2024-04-04,1,1,900,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
4,2024-04-05,1,2,1800,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
...,...,...,...,...,...,...,...,...,...,...
1987,2025-03-27,6,2,1800,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
1988,2025-03-28,6,3,2700,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
1989,2025-03-29,6,1,900,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966
1990,2025-03-31,6,2,1800,1,1,9,False,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966


In [35]:
to_insert = df_long[
    ['product_id', 'creator_id', 'update_id',
     'date', 'weather_history_id',
     'sales_count', 'sales_revenue', "created_at","updated_at",'is_deleted']
]

In [42]:
to_insert = to_insert.drop_duplicates(
    subset=['product_id', 'date'],
    keep='last'
).reset_index(drop=True)

In [43]:
to_insert

,product_id,creator_id,update_id,date,weather_history_id,sales_count,sales_revenue,created_at,updated_at,is_deleted
0,1,1,1,2024-04-01,9,4,3600,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
1,1,1,1,2024-04-02,9,3,2700,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
2,1,1,1,2024-04-03,9,2,1800,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
3,1,1,1,2024-04-04,9,1,900,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
4,1,1,1,2024-04-05,9,2,1800,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
...,...,...,...,...,...,...,...,...,...,...
1879,6,1,1,2025-03-27,9,2,1800,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
1880,6,1,1,2025-03-28,9,3,2700,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
1881,6,1,1,2025-03-29,9,1,900,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False
1882,6,1,1,2025-03-31,9,2,1800,2025-06-23 17:27:07.441966,2025-06-23 17:27:07.441966,False


In [44]:
sql = """
INSERT INTO sales_history
  (product_id, creator_id, update_id,
   date, weather_history_id,
   sales_count, sales_revenue,created_at, updated_at,
   is_deleted)
VALUES %s
ON CONFLICT (product_id, date) DO UPDATE
  SET
    sales_count   = EXCLUDED.sales_count,
    sales_revenue = EXCLUDED.sales_revenue,
    is_deleted    = EXCLUDED.is_deleted,
    updated_at    = EXCLUDED.updated_at
"""

In [45]:
conn.rollback()

In [46]:
import psycopg2
from psycopg2.extras import execute_values
records = to_insert.values.tolist()
execute_values(cur, sql, records)

In [47]:
conn.commit()